# 📥 01_download_designated_places.ipynb
================================================================================

**Script 01 of 08** in the Networks Paper analysis pipeline.

## 🎯 Purpose
Download U.S. Census-designated place (CDP) shapefiles and demographic data, then join them together. This creates the foundational dataset of community boundaries that all subsequent scripts depend on.

## 🔥 Why This Matters for Wildfire Research
Census Designated Places represent communities - the places where people live and may need to evacuate during wildfires. By downloading both geographic boundaries AND demographic data, we can later analyze:
- **Evacuation capacity** vs. population size
- **Vulnerable populations** (elderly, disabled) vs. egress routes
- **Vehicle access** vs. road network connectivity

## 📋 Workflow
1. **Download shapefiles** from Census TIGER/Line for all 50 states + DC
2. **Fetch demographics** from Census ACS API (population, age, income, vehicles, etc.)
3. **Join** demographics to shapefile attributes using FIPS codes
4. **Filter** to places with population < 50,000 (focus on smaller communities)
5. **Save** updated shapefiles per state

---

## 📥 INPUTS (Data Sources)
| Source | URL/Path | Description |
|--------|----------|-------------|
| TIGER/Line Shapefiles | `https://www2.census.gov/geo/tiger/TIGER2023/PLACE/` | Census place boundaries (2023) |
| Census ACS API | `https://api.census.gov/data/2021/acs/acs5` | 5-year American Community Survey demographics |
| Census API Key | Environment variable `CENSUS_API_KEY` | Required for API access - get at census.gov |

## 📤 OUTPUTS (Generated Files)
| Output | Path | Description |
|--------|------|-------------|
| State Shapefiles | `.../us-census-designated-places/{State}_{FIPS}/tl_2023_{FIPS}_place.shp` | Place boundaries with demographics |
| Demographics CSV | `.../us-census-designated-places/{State}_{FIPS}/{State}_{FIPS}_demographics.csv` | Raw demographics per state |

## ⏱️ Expected Runtime
- ~30-60 minutes for all 50 states (parallel processing)
- Depends on Census API response times

---

## ⚙️ SETTINGS

In [ ]:
import os
import requests
from zipfile import ZipFile
from concurrent.futures import ProcessPoolExecutor, as_completed
import glob
import json
import logging
import csv
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
from datetime import datetime
import pytz

################################################################################
#                            🔧 SETTINGS & CONFIGURATION                       #
################################################################################

# =============================================================================
# ⏰ Timezone Configuration (Pacific Time for timestamps)
# =============================================================================
TIMEZONE = pytz.timezone('America/Los_Angeles')

def print_timestamped(message):
    """Print message with Pacific Time timestamp."""
    timestamp = datetime.now(TIMEZONE).strftime('%Y-%m-%d %H:%M:%S %Z')
    print(f"[{timestamp}] {message}")

# =============================================================================
# 📝 Logging Configuration
# =============================================================================
logging.basicConfig(
    level=logging.INFO,
    format='[%(levelname)s] %(asctime)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# =============================================================================
# 🔑 API Key Configuration
# =============================================================================
# Get your key at: https://api.census.gov/data/key_signup.html
CENSUS_API_KEY = os.getenv("CENSUS_API_KEY", "bdc7dcc15aba822045f9fff8a66d048f90a8a973")
if not CENSUS_API_KEY:
    logger.error("⚠️ CENSUS_API_KEY is not set. Please set it before running the script.")
    exit(1)

# =============================================================================
# 🖥️ Parallel Processing Settings
# =============================================================================
NUM_WORKERS = 10  # Number of parallel workers for downloading

################################################################################
#                            📂 FILE PATHS                                      #
################################################################################
# ┌─────────────────────────────────────────────────────────────────────────┐
# │  Configure BASE_DATA_DIR to point to your root data directory.          │
# │  You can also override it at runtime with an environment variable:      │
# │      export NETWORKS_DATA_DIR="/your/path/to/data"                      │
# │  Or edit the default path below.                                        │
# └─────────────────────────────────────────────────────────────────────────┘
BASE_DATA_DIR = os.environ.get("NETWORKS_DATA_DIR", os.path.expanduser("~/data/networks_paper"))

# =============================================================================
# 📤 Output Directory
# =============================================================================
output_dir = os.path.join(BASE_DATA_DIR, "us-census-designated-places")
os.makedirs(output_dir, exist_ok=True)
logger.info(f"📂 Output directory: {output_dir}")

# =============================================================================
# 🌐 Data Source URLs
# =============================================================================
# Base URL for TIGER/Line 2023 place shapefiles
base_url = "https://www2.census.gov/geo/tiger/TIGER2023/PLACE/"

# Census ACS 5-year endpoint (2021)
census_data_base_url = "https://api.census.gov/data/2021/acs/acs5"

################################################################################
#                            📊 CENSUS VARIABLES                               #
################################################################################

# =============================================================================
# 📊 Core Variables (Population & Households)
# =============================================================================
# B01003_001E = Total Population
# B11001_001E = Number of Households
core_variables = ["B01003_001E", "B11001_001E"]

# =============================================================================
# 👴 Age 65+ Variables (Senior Population)
# =============================================================================
# Male Over 65: B01001_020E (65-66), B01001_021E (67-69), B01001_022E (70-74), 
#               B01001_023E (75-79), B01001_024E (80-84), B01001_025E (85+)
# Female Over 65: B01001_044E (65-66), B01001_045E (67-69), B01001_046E (70-74), 
#                 B01001_047E (75-79), B01001_048E (80-84), B01001_049E (85+)
age_65_plus_variables = [
    "B01001_020E", "B01001_021E", "B01001_022E", "B01001_023E", "B01001_024E", "B01001_025E",
    "B01001_044E", "B01001_045E", "B01001_046E", "B01001_047E", "B01001_048E", "B01001_049E"
]

# =============================================================================
# 💰 Additional Variables (Wealth, Disability, Vehicle Access, Commute)
# =============================================================================
additional_variables = [
    # Wealth
    "B19301_001E",  # Per Capita Income
    "B19013_001E",  # Median Household Income
    "B17001_002E",  # Total Population Below Poverty Level

    # Economic Activity
    "B23025_001E",  # Employment Status - Total
    "B23025_002E",  # Employment Status - Employed
    "B23025_003E",  # Employment Status - Unemployed

    # Disability
    "B18101_001E",  # Total Population with Disability
    "B18110_002E",  # Hearing Disability
    "B18110_003E",  # Vision Disability
    "B18110_004E",  # Cognitive Disability
    "B18110_005E",  # Ambulatory Disability
    "B18110_006E",  # Self-care Disability
    "B18110_007E",  # Independent Living Disability

    # Access to Vehicle
    "B08201_002E",  # Households with Vehicles
    "B08201_010E",  # Households without Vehicles
    "B08201_007E",  # Workers Using Public Transport

    # Commute Time Categories
    "B08301_001E",  # Total Commute Time
    "B08301_002E",  # Less than 5 minutes
    "B08301_003E",  # 5 to 9 minutes
    "B08301_004E",  # 10 to 14 minutes
    "B08301_005E",  # 15 to 19 minutes
    "B08301_006E",  # 20 to 24 minutes
    "B08301_007E",  # 25 to 29 minutes
    "B08301_008E",  # 30 to 34 minutes
    "B08301_009E",  # 35 to 39 minutes
    "B08301_010E",  # 40 to 44 minutes
    "B08301_011E",  # 45 to 59 minutes
    "B08301_012E"   # 60 minutes or more
]

# Combine all variables
variables = core_variables + age_65_plus_variables + additional_variables

# =============================================================================
# 🏷️ Variable Name Mapping (ACS codes to descriptive names)
# =============================================================================
header_mapping = {
    "NAME": "Name",
    "B01003_001E": "Total Population",
    "B11001_001E": "Number of Households",
    "B01001_020E": "Male 65-66",
    "B01001_021E": "Male 67-69",
    "B01001_022E": "Male 70-74",
    "B01001_023E": "Male 75-79",
    "B01001_024E": "Male 80-84",
    "B01001_025E": "Male 85+",
    "B01001_044E": "Female 65-66",
    "B01001_045E": "Female 67-69",
    "B01001_046E": "Female 70-74",
    "B01001_047E": "Female 75-79",
    "B01001_048E": "Female 80-84",
    "B01001_049E": "Female 85+",
    "state": "State FIPS",
    "place": "Place FIPS",

    # Wealth
    "B19301_001E": "Per Capita Income",
    "B19013_001E": "Median Household Income",
    "B17001_002E": "Total Population Below Poverty Level",

    # Economic Activity
    "B23025_001E": "Employment Status - Total",
    "B23025_002E": "Employment Status - Employed",
    "B23025_003E": "Employment Status - Unemployed",

    # Disability
    "B18101_001E": "Total Population with Disability",
    "B18110_002E": "Hearing Disability",
    "B18110_003E": "Vision Disability",
    "B18110_004E": "Cognitive Disability",
    "B18110_005E": "Ambulatory Disability",
    "B18110_006E": "Self-care Disability",
    "B18110_007E": "Independent Living Disability",

    # Access to Vehicle
    "B08201_002E": "Households with Vehicles",
    "B08201_010E": "Households without Vehicles",
    "B08201_007E": "Workers Using Public Transport",

    # Commute Time Categories
    "B08301_001E": "Total Commute Time",
    "B08301_002E": "Commute Time < 5 minutes",
    "B08301_003E": "Commute Time 5-9 minutes",
    "B08301_004E": "Commute Time 10-14 minutes",
    "B08301_005E": "Commute Time 15-19 minutes",
    "B08301_006E": "Commute Time 20-24 minutes",
    "B08301_007E": "Commute Time 25-29 minutes",
    "B08301_008E": "Commute Time 30-34 minutes",
    "B08301_009E": "Commute Time 35-39 minutes",
    "B08301_010E": "Commute Time 40-44 minutes",
    "B08301_011E": "Commute Time 45-59 minutes",
    "B08301_012E": "Commute Time 60 minutes or more"
}

################################################################################
#                            🗺️ STATE FIPS CODES                              #
################################################################################

# =============================================================================
# 🏛️ State FIPS Code to Name Mapping (all 50 states + DC)
# =============================================================================
state_fips_names = {
    "01": "Alabama", "02": "Alaska", "04": "Arizona", "05": "Arkansas", "06": "California",
    "08": "Colorado", "09": "Connecticut", "10": "Delaware", "11": "District_of_Columbia", "12": "Florida",
    "13": "Georgia", "15": "Hawaii", "16": "Idaho", "17": "Illinois", "18": "Indiana",
    "19": "Iowa", "20": "Kansas", "21": "Kentucky", "22": "Louisiana", "23": "Maine",
    "24": "Maryland", "25": "Massachusetts", "26": "Michigan", "27": "Minnesota", "28": "Mississippi",
    "29": "Missouri", "30": "Montana", "31": "Nebraska", "32": "Nevada", "33": "New_Hampshire",
    "34": "New_Jersey", "35": "New_Mexico", "36": "New_York", "37": "North_Carolina", "38": "North_Dakota",
    "39": "Ohio", "40": "Oklahoma", "41": "Oregon", "42": "Pennsylvania", "44": "Rhode_Island",
    "45": "South_Carolina", "46": "South_Dakota", "47": "Tennessee", "48": "Texas", "49": "Utah",
    "50": "Vermont", "51": "Virginia", "53": "Washington", "54": "West_Virginia", "55": "Wisconsin",
    "56": "Wyoming"
}
logger.info(f"📊 Total states to process: {len(state_fips_names)}")

################################################################################
#                            🛠️ HELPER FUNCTIONS                               #
################################################################################

@retry(
    wait=wait_exponential(multiplier=1, min=4, max=10),
    stop=stop_after_attempt(5),
    retry=retry_if_exception_type(requests.exceptions.RequestException),
    reraise=True
)
def fetch_url(url, params=None, timeout=120):
    print("Fetching URL:", url)
    response = requests.get(url, params=params, timeout=timeout)
    response.raise_for_status()
    return response

def download_and_extract(fips_code, state_name):
    """
    Download shapefile for a state, fetch demographics from Census API, and join them.
    
    Args:
        fips_code (str): 2-digit state FIPS code
        state_name (str): State name for directory naming
    """
    # =========================================================================
    # 📥 DOWNLOAD AND EXTRACT SHAPEFILE
    # =========================================================================
    print_timestamped(f"Starting download_and_extract for {state_name}")
    logger.info(f"Starting download for {state_name} (FIPS code {fips_code})...")
    url = f"{base_url}tl_2023_{fips_code}_place.zip"
    try:
        response = fetch_url(url)
        logger.info(f"Downloading shapefile for {state_name} (FIPS code {fips_code}) from {url}")
        
        state_dir = os.path.join(output_dir, f"{state_name}_{fips_code}")
        os.makedirs(state_dir, exist_ok=True)
        logger.info(f"Directory created for {state_name}: {state_dir}")

        zip_path = os.path.join(state_dir, f"tl_2023_{fips_code}_place.zip")
        with open(zip_path, "wb") as f:
            print("Writing ZIP for", state_name)
            f.write(response.content)
        logger.info(f"ZIP file saved for {state_name} at {zip_path}")

        with ZipFile(zip_path, 'r') as z:
            print("Extracting ZIP for", state_name)
            z.extractall(state_dir)
        logger.info(f"ZIP file extracted for {state_name}.")

        extracted_files = glob.glob(os.path.join(state_dir, "*"))
        logger.info(f"Extracted {len(extracted_files)} files for {state_name} to {state_dir}")
        if len(extracted_files) == 0:
            logger.warning(f"No files extracted for {state_name}. Check if the ZIP was empty or malformed.")

        # =====================================================================
        # 📊 DOWNLOAD CENSUS DEMOGRAPHIC DATA
        # =====================================================================
        download_census_data_for_state(fips_code, state_name, state_dir)

        # =====================================================================
        # 🔗 JOIN DEMOGRAPHICS WITH SHAPEFILE
        # =====================================================================
        join_demographics_with_shapefile(state_fips=fips_code, state_name=state_name, state_dir=state_dir)

    except requests.exceptions.HTTPError as http_err:
        logger.error(f"HTTP error occurred while downloading shapefile for {state_name} (FIPS {fips_code}): {http_err}")
    except requests.exceptions.RequestException as req_err:
        logger.error(f"Request exception for {state_name} (FIPS {fips_code}): {req_err}")
    except Exception as e:
        logger.error(f"Exception occurred while processing {state_name} (FIPS {fips_code}): {e}")

def join_demographics_with_shapefile(state_fips, state_name, state_dir):
    """
    Join Census demographics CSV with shapefile attributes and filter by population.
    
    Args:
        state_fips (str): 2-digit state FIPS code
        state_name (str): State name
        state_dir (str): Path to state directory containing shapefile and CSV
    """
    # =========================================================================
    # 🔗 JOIN DEMOGRAPHICS WITH SHAPEFILE
    # =========================================================================
    print_timestamped(f"Starting join_demographics_with_shapefile for {state_name}")
    try:
        shapefile_pattern = os.path.join(state_dir, "tl_2023_*_place.shp")
        shapefile_paths = glob.glob(shapefile_pattern)
        if not shapefile_paths:
            logger.error(f"No shapefile found in {state_dir} matching pattern {shapefile_pattern}")
            return
        shapefile_path = shapefile_paths[0]
        logger.info(f"Shapefile found: {shapefile_path}")

        # Read shapefile
        print("Reading shapefile for", state_name)
        gdf = gpd.read_file(shapefile_path)
        logger.info(f"Shapefile read into GeoDataFrame with {len(gdf)} records.")

        # Read CSV
        csv_path = os.path.join(state_dir, f"{state_name}_{state_fips}_demographics.csv")
        if not os.path.exists(csv_path):
            logger.error(f"CSV file not found: {csv_path}")
            return
        print("Reading CSV for", state_name)
        df = pd.read_csv(csv_path)
        logger.info(f"CSV file read into DataFrame with {len(df)} records.")

        # Zero-pad FIPS
        print("Adjusting FIPS codes for join", state_name)
        gdf['State FIPS'] = gdf['STATEFP'].astype(str).str.zfill(2)
        gdf['Place FIPS'] = gdf['PLACEFP'].astype(str).str.zfill(5)
        df['State FIPS'] = df['State FIPS'].astype(str).str.zfill(2)
        df['Place FIPS'] = df['Place FIPS'].astype(str).str.zfill(5)

        # Create join key
        gdf['GEOID_join'] = gdf['State FIPS'] + gdf['Place FIPS']
        df['GEOID_join'] = df['State FIPS'] + df['Place FIPS']

        # Identify demographic columns
        demographic_columns = [c for c in df.columns if c in header_mapping.values() or c == "GEOID_join"]

        # Merge
        print("Merging data for", state_name)
        merged_gdf = gdf.merge(df[demographic_columns], on='GEOID_join', how='left')
        logger.info(f"GeoDataFrame after merge has {len(merged_gdf)} records.")

        # Check missing
        missing_demographics = merged_gdf['Total Population'].isnull().sum()
        if missing_demographics > 0:
            logger.warning(f"{missing_demographics} records have missing demographic data.")

        # Filter to only keep places that have < 50,000 Total Population
        print("Filtering CDPs with Total Population < 50000 for", state_name)
        merged_gdf['Total Population'] = pd.to_numeric(merged_gdf['Total Population'], errors='coerce')
        merged_gdf = merged_gdf[merged_gdf['Total Population'] < 50000]

        # Drop join key
        if 'GEOID_join' in merged_gdf.columns:
            merged_gdf = merged_gdf.drop(columns=['GEOID_join'])

        # Save updated shapefile
        output_shapefile_path = os.path.join(state_dir, f"tl_2023_{state_fips}_place.shp")
        print("Saving updated shapefile for", state_name)
        merged_gdf.to_file(output_shapefile_path)
        logger.info(f"Updated shapefile with demographics saved at {output_shapefile_path}")

    except Exception as e:
        logger.error(f"Exception occurred while joining demographics with shapefile for {state_name} (FIPS {state_fips}): {e}")

def chunk_list(lst, n):
    """Yield successive n-sized chunks from lst."""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

@retry(
    wait=wait_exponential(multiplier=1, min=4, max=10),
    stop=stop_after_attempt(5),
    retry=retry_if_exception_type(requests.exceptions.RequestException),
    reraise=True
)
def download_census_data_for_state(state_fips, state_name, state_dir):
    """
    Download Census ACS demographic data for a state in chunks and save to CSV.
    
    Args:
        state_fips (str): 2-digit state FIPS code
        state_name (str): State name
        state_dir (str): Path to state directory for saving CSV
    """
    # =========================================================================
    # 📊 DOWNLOAD CENSUS DATA IN CHUNKS (API limits)
    # =========================================================================
    print_timestamped(f"Starting download_census_data_for_state for {state_name}")
    logger.info(f"Fetching Census data for {state_name} (FIPS {state_fips}) in chunks...")

    # Define chunk size (number of variables per request). Adjust as needed.
    chunk_size = 10

    main_df = None
    first_chunk = True
    for chunk in chunk_list(variables, chunk_size):
        # For the first chunk, include NAME, state, and place to serve as keys
        if first_chunk:
            query_fields = ["NAME"] + chunk + ["state", "place"]
            first_chunk = False
        else:
            # For subsequent chunks, include state and place to allow merge
            query_fields = chunk + ["state", "place"]
        
        var_str = ",".join(query_fields)
        params = {
            "get": var_str,
            "for": "place:*",
            "in": f"state:{state_fips}",
            "key": CENSUS_API_KEY
        }
        
        try:
            response = fetch_url(census_data_base_url, params=params)
            print("Parsing JSON for", state_name, "with variables:", chunk)
            data = response.json()
            headers = data[0]
            rows = data[1:]
            df_chunk = pd.DataFrame(rows, columns=headers)
            
            if main_df is None:
                main_df = df_chunk
            else:
                # Merge on 'state' and 'place'
                main_df = pd.merge(main_df, df_chunk, on=["state", "place"], how="outer", suffixes=("", "_dup"))
                # Remove any duplicate columns that might have arisen
                duplicate_cols = [col for col in main_df.columns if col.endswith("_dup")]
                main_df.drop(columns=duplicate_cols, inplace=True)
            
            logger.info(f"Fetched and merged chunk with variables: {chunk}")
        
        except Exception as e:
            logger.error(f"Exception occurred while fetching Census chunk for {state_name} (FIPS {state_fips}): {e}")
    
    if main_df is not None:
        # Apply header mapping to rename columns as desired
        main_df.rename(columns=header_mapping, inplace=True)
        # Save merged data to CSV
        output_csv_path = os.path.join(state_dir, f"{state_name}_{state_fips}_demographics.csv")
        main_df.to_csv(output_csv_path, index=False)
        logger.info(f"Saved demographic data for {state_name} to {output_csv_path}")
    else:
        logger.error(f"No data fetched for {state_name} (FIPS {state_fips}).")

################################################################################
#                            🚀 MAIN EXECUTION                                 #
################################################################################

print_timestamped("🚀 Starting main process...")
logger.info("Preparing to start parallel processing.")
tasks = [(fips_code, state_name) for fips_code, state_name in state_fips_names.items()]
logger.info("Submitting tasks to the executor.")

with ProcessPoolExecutor(max_workers=10) as executor:
    future_to_task = {executor.submit(download_and_extract, fips, name): (fips, name) for fips, name in tasks}
    for future in tqdm(as_completed(future_to_task), total=len(future_to_task), desc="Processing States"):
        fips, name = future_to_task[future]
        try:
            future.result()
            logger.info(f"Completed task for {name} (FIPS code {fips})")
        except Exception as exc:
            logger.error(f"{name} (FIPS code {fips}) generated an exception: {exc}")

logger.info("All tasks have been processed. Check the output directory for the extracted state folders and demographic CSV files.")
